# 02 — Base empresa-ano (Fase 2)

Este notebook transforma a base trimestral do Economatica na base de modelagem: **uma linha por empresa × ano**, com o alvo (entrada em distress nos dois anos seguintes), as explicativas, o Z''-score de comparação, a divisão temporal e a winsorização.

Cada passo chama uma função de `src/base.py`, então a mesma transformação pode ser refeita de ponta a ponta com `base.construir()`. As escolhas (anos, divisão, limites, exclusões) estão em `src/config.py`.

Decisões da dupla aplicadas aqui (21/10/2026):
- **H5 abandonada**: a dívida em moeda estrangeira só existe em 2010;
- **as 8 financeiras excluídas**, incluindo as corretoras de seguro;
- **setor provisório** a partir da CVM, até chegar o Subsetor Bovespa;
- **anos anteriores ao registro na CVM mantidos**, com flag;
- **divisão**: treino 2010–2018, validação 2019–2020, teste 2021–2023.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config as cfg, dados, base, style

style.aplicar()
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: style.num_br(x, 3))

## 1. Só o 4º trimestre

Ficamos com o 4T de cada ano, por três motivos:
1. **Resolve as variáveis acumuladas.** No 4T, "no exercício" é o ano completo, então a receita e a despesa financeira passam a cobrir o mesmo período que o EBIT, o lucro e o FCO, que são de 12 meses (Fase 1, seção 3a).
2. **O alvo é anual.** "Distress nos dois anos seguintes" é uma afirmação sobre anos, e o balanço anual é auditado.
3. **Evita dependência artificial.** Quatro trimestres de 12 meses móveis da mesma empresa são quase a mesma informação repetida.

Também saem as colunas que a Fase 1 mostrou serem redundantes ou inúteis:
- `passivo_mais_pl` (igual ao ativo total);
- `liquidez_corrente` (recalculada);
- `ebitda_desp_fin` (usa EBITDA);
- `divida_moeda_estrangeira` (H5 abandonada);
- `juros_pagos` e `capex_12m` (vazias).

Linhas sem ativo total são anos em que a empresa não está no painel e também saem.

In [2]:
trim = dados.carregar_economatica()
ea = base.empresa_ano(trim)
print('empresa-anos (4T com balanço):', len(ea), '| empresas:', ea['ticker'].nunique(), '| anos:', ea['ano'].min(), '–', ea['ano'].max())

empresa-anos (4T com balanço): 3712 | empresas: 291 | anos: 2010 – 2025


## 2. Financeiras

A Fase 1 (seção 3f) encontrou 8 empresas classificadas pela CVM em setores financeiros. A dupla decidiu excluir todas. Bancos, seguradoras e holdings financeiras têm estrutura de balanço própria: a dívida é a matéria-prima do negócio, e a despesa financeira é custo operacional. Por isso alavancagem e cobertura de juros não significam a mesma coisa para elas.

In [3]:
fin = ea['ticker'].isin(cfg.EXCLUIR_FINANCEIRAS)
print('Empresa-anos excluídas:', int(fin.sum()), '| empresas:', sorted(ea.loc[fin, 'ticker'].unique()))
print('(B3SA3 não tem nenhum balanço, então já não estava na base empresa-ano)')
ea = ea[~fin].copy()
print('Restam:', len(ea), 'empresa-anos de', ea['ticker'].nunique(), 'empresas')

Empresa-anos excluídas: 79 | empresas: ['B1003', 'BRBI11', 'CXSE3', 'HBTS5', 'ITSA4', 'QUAL3', 'WIZC3']
(B3SA3 não tem nenhum balanço, então já não estava na base empresa-ano)
Restam: 3633 empresa-anos de 284 empresas


## 3. Cadastro da CVM

A base do Economatica não tem CNPJ. A ligação é feita pelo **código de negociação** no FCA da CVM, em três etapas (exato → raiz de 4 letras → tabela manual), como explicado na Fase 1. Do cadastro trazemos a situação, a primeira data de registro, a data e o motivo do cancelamento e o setor.

**Setor provisório:** o `SETOR_ATIV` da CVM tem 47 categorias e separa as holdings ("Emp. Adm. Part. - Energia Elétrica") das operacionais do mesmo setor. Juntamos a holding com o setor dela e agrupamos em 10 macrossetores, inspirados nos setores econômicos da B3 (`cfg.SETOR_MACRO`). Esse agrupamento será trocado pelo Subsetor Bovespa quando a dupla exportá-lo do Economatica.

In [4]:
cad1 = dados.cad_por_cnpj(dados.carregar_cad_cvm())
mapa = dados.mapa_ticker_cnpj(sorted(ea['ticker'].unique()))
ea = base.juntar_cvm(ea, mapa, cad1)

emp = ea.drop_duplicates('ticker')
print('Empresas casadas com a CVM:', int(emp['CNPJ_CIA'].notna().sum()), 'de', len(emp))
display(emp['como_casou'].value_counts().to_frame('empresas'))
print('Sem CNPJ:', emp.loc[emp['CNPJ_CIA'].isna(), 'ticker'].tolist(),
      '| empresa-anos afetados:', int(ea['CNPJ_CIA'].isna().sum()))

Empresas casadas com a CVM: 282 de 284


,empresas
como_casou,
exato,262
manual,13
raiz,6
sem_match,2
manual_conferir,1


Sem CNPJ: ['AURA33', 'LLBI3'] | empresa-anos afetados: 19


In [5]:
# Macrossetor provisório
pd.crosstab(emp['setor'], 'empresas').sort_values('empresas', ascending=False)

col_0,empresas
setor,
Consumo cíclico,61
Bens industriais,43
Construção e imobiliário,39
Utilidade pública,39
Materiais básicos,31
Consumo não cíclico,23
Comunicações e TI,18
Saúde,14
Holdings e outros,7


In [6]:
# Idade de registro e anos anteriores ao registro (prospecto)
print('Empresa-anos com t anterior ao registro na CVM (pre_registro):', int(ea['pre_registro'].sum()),
      'de', ea['ticker'][ea['pre_registro']].nunique(), 'empresas')
ea.groupby('ano')['pre_registro'].sum().astype(int).to_frame('pré-registro').T

Empresa-anos com t anterior ao registro na CVM (pre_registro): 217 de 79 empresas


ano,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
pré-registro,8,5,8,4,7,10,15,36,43,43,28,3,2,3,2,0


**Leitura.** Os anos anteriores ao registro na CVM se concentram em 2016–2020. São as demonstrações que as empresas da onda de IPOs de 2020–21 publicaram no prospecto. A flag `pre_registro` permite, na Fase 4, testar se o resultado muda quando esses anos saem.

## 4. Alvo

**Definições** (EBIT aparece só aqui e no Z'', nunca nas explicativas):
- `cobertura_t = EBIT_t / despesa financeira_t`. Se a despesa for menor ou igual a zero, a cobertura é **NaN** ("não classificável"): a despesa negativa vem de ganho cambial ou de reclassificação, e a razão perderia o sentido.
- `distress_t = 1` se `cobertura_t < 1`, ou seja, o resultado operacional não paga a despesa financeira. EBIT negativo também conta.
- `Y_t = 1` se `distress_{t+1} = 1` **e** `distress_{t+2} = 1`. `Y_t = 0` se os dois anos são classificáveis e não estão ambos em distress. Nos outros casos `Y_t` é NaN.
- **Amostra em risco:** `cobertura_t ≥ 1` (a empresa ainda não está em distress em t, então medimos a **entrada**) e `Y_t` observado. Como o alvo usa t+2 e o último ano completo é 2025, t vai de 2010 a 2023.

Exigir **dois anos seguidos** de distress filtra os tropeços pontuais, como um ano ruim isolado ou uma despesa financeira inflada por variação cambial, e captura a dificuldade persistente que importa para crédito.

O t+1 e o t+2 são procurados **por ano**, e não pela "próxima linha". Se uma empresa pula um ano no painel, o alvo dela fica NaN em vez de usar o ano errado.

In [7]:
ea = base.adicionar_alvo(ea)
base.funil_amostra(ea)

,etapa,linhas,perdidas na etapa
0,"empresa-anos com balanço, t = 2010–2023",3085,0
1,cobertura classificável em t,2970,115
2,fora de distress em t (cobertura ≥ 1),1742,1228
3,empresa presente em t+1 e t+2,1730,12
4,t+1 e t+2 classificáveis (Y observado),1708,22


In [8]:
# Distress em t (sem condicionar a nada): qual parcela das empresas tem cobertura < 1 em cada ano?
d = ea[ea['distress'].notna()]
d.groupby('ano')['distress'].agg(['mean', 'size']).rename(columns={'mean': '% com cobertura < 1', 'size': 'classificáveis'}).T

ano,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
% com cobertura < 1,"0,268","0,345","0,392","0,333","0,423","0,591","0,606","0,473","0,409","0,394","0,443","0,303","0,403","0,401","0,434","0,414"
classificáveis,"168,000","168,000","176,000","177,000","182,000","186,000","198,000","222,000","232,000","241,000","246,000","254,000","258,000","262,000","267,000","263,000"


**Leitura do funil.**
- **Cobertura não classificável** (115 linhas): despesa financeira menor ou igual a zero, ou EBIT ou despesa ausentes.
- **Já em distress em t** (1.228 linhas, 41%): é o maior corte. A cobertura menor que 1 é **comum** na base, atingindo de 27% a 60% das empresas conforme o ano, com pico na recessão de 2015–16. Dois motivos: a despesa financeira do Economatica é **bruta** (inclui variação cambial e monetária, não só juros), e a Selic brasileira é alta. Por isso a definição de distress é exigente, e a exigência de dois anos seguidos de distress no alvo é importante.
- **A empresa some em t+1 ou t+2** (12 linhas): são os candidatos a **eventos perdidos** (tabela abaixo).
- **t+1 ou t+2 não classificáveis** (22 linhas).

In [9]:
# Empresas que pararam de reportar: fora de distress em t, mas sem balanço em t+1 ou t+2
sumiu = ea[ea['ano'].between(cfg.ANO_INI, cfg.ANO_T_MAX) & (ea['cobertura'] >= 1) & ~(ea['presente_t1'] & ea['presente_t2'])]
print('Linhas perdidas porque a empresa sumiu do painel:', len(sumiu), '| empresas:', sumiu['ticker'].nunique())
sumiu[['ticker', 'ano', 'DENOM_SOCIAL', 'SIT', 'DT_CANCEL', 'presente_t1', 'presente_t2', 'cobertura']]

Linhas perdidas porque a empresa sumiu do painel: 12 | empresas: 7


,ticker,ano,DENOM_SOCIAL,SIT,DT_CANCEL,presente_t1,presente_t2,cobertura
221,ARML3,2021,"ARMAC LOCAÇÃO, LOGÍSTICA E SERVIÇOS S.A.",ATIVO,NaT,True,False,"1,916"
222,ARML3,2022,"ARMAC LOCAÇÃO, LOGÍSTICA E SERVIÇOS S.A.",ATIVO,NaT,False,True,"1,147"
801,CRPG5,2016,TRONOX PIGMENTOS DO BRASIL S.A.,ATIVO,NaT,False,False,"2,836"
1042,DOHL4,2020,DOHLER S.A.,ATIVO,NaT,False,False,"4,316"
1342,FASA3,2022,FASA S.A. - EM RECUPERAÇÃO JUDICIAL,ATIVO,NaT,True,False,"9,995"
1455,GEPA4,2022,RIO PARANAPANEMA ENERGIA SA,ATIVO,NaT,False,False,"2,398"
1531,GPAR3,2020,COMPANHIA CELG DE PARTICIPAÇÕES - CELGPAR,ATIVO,NaT,True,False,"33,903"
1532,GPAR3,2021,COMPANHIA CELG DE PARTICIPAÇÕES - CELGPAR,ATIVO,NaT,False,False,"13,227"
3268,TIMS3,2019,TIM S.A.,ATIVO,NaT,True,False,"3,016"
3269,TIMS3,2020,TIM S.A.,ATIVO,NaT,False,True,"1,755"


**Leitura.** Só 12 linhas, de 7 empresas, se perdem assim, e nenhuma delas cancelou o registro na CVM. São dois tipos de caso:
- **Buracos no painel:** falta o 4T de algum ano no Economatica, mas a empresa volta depois. É o caso de TIM (TIMS3), Armac (ARML3) e CELGPAR (GPAR3). Vale conferir no Economatica se o dado existe.
- **A empresa para de aparecer:** Tronox (CRPG5, depois de 2016), Dohler (DOHL4, depois de 2020), Rio Paranapanema (GEPA4, depois de 2022) e **Fasa (FASA3), em recuperação judicial**, que é o candidato mais claro a **evento de distress perdido**.

Poucas linhas se perdem porque o *Screening* praticamente só trouxe empresas que **sobreviveram** até hoje (Fase 1, seção 3g). **O problema maior está nas empresas que nunca entraram na base**, e ele é medido na Fase 3.

In [10]:
risco = ea[ea['em_risco']]
print(f"Amostra em risco: {len(risco)} empresa-anos | {risco['ticker'].nunique()} empresas | "
      f"eventos (Y=1): {int(risco['Y'].sum())} | taxa: {style.pct_br(risco['Y'].mean())}")
por_ano = risco.groupby('ano')['Y'].agg(empresa_anos='size', eventos='sum', taxa='mean')
por_ano['eventos'] = por_ano['eventos'].astype(int)
por_ano.T

Amostra em risco: 1708 empresa-anos | 246 empresas | eventos (Y=1): 197 | taxa: 11,5%


ano,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
empresa_anos,"122,000","110,000","105,000","114,000","102,000","76,000","76,000","116,000","136,000","144,000","132,000","174,000","147,000","154,000"
eventos,"18,000","11,000","3,000","18,000","28,000","13,000","9,000","11,000","11,000","14,000","7,000","27,000","12,000","15,000"
taxa,"0,148","0,100","0,029","0,158","0,275","0,171","0,118","0,095","0,081","0,097","0,053","0,155","0,082","0,097"


In [11]:
por_setor = risco.groupby('setor')['Y'].agg(empresa_anos='size', eventos='sum', taxa='mean').sort_values('taxa', ascending=False)
por_setor['eventos'] = por_setor['eventos'].astype(int)
por_setor

,empresa_anos,eventos,taxa
setor,,,
Materiais básicos,182,30,"0,165"
Bens industriais,237,36,"0,152"
Consumo cíclico,291,38,"0,131"
Saúde,77,9,"0,117"
Construção e imobiliário,285,31,"0,109"
Consumo não cíclico,117,12,"0,103"
Comunicações e TI,97,9,"0,093"
Petróleo e gás,36,3,"0,083"
Utilidade pública,326,25,"0,077"


## 5. Explicativas em t

Todas são **índices**, para que empresas de tamanhos diferentes fiquem comparáveis. Nenhuma usa EBIT ou EBITDA, porque eles estão no alvo e usá-los seria explicar a cobertura com ela mesma.

| variável | definição | hipótese |
|---|---|---|
| `alavancagem` | dívida bruta / ativo total | H1 (+): mais dívida, mais juros a pagar |
| `divida_cp_share` | empréstimos CP / (CP + LP) | H1 (+): dívida concentrada no curto prazo exige rolagem |
| `roa` | lucro líquido 12m / ativo total | H2 (−): empresa rentável tem folga |
| `liquidez` | caixa / passivo circulante | H3 (−): caixa para honrar o curto prazo |
| `fco_divida` | FCO 12m / dívida bruta | H4 (−): capacidade de pagar a dívida com a operação |
| `log_ativo` | log do ativo total | controle de tamanho |
| `setor`, `ano` | macrossetor (provisório) e ano | controles |
| `idade_registro` | t − ano de registro na CVM | só descritiva |

Detalhes:
- Denominador zero ou ausente vira NaN, nunca infinito.
- Empresas **sem dívida** (`sem_divida = True`) ficam com `fco_divida` e `divida_cp_share` NaN, porque a razão não existe. A flag guarda essa informação para a modelagem.
- A liquidez é recalculada como caixa / PC. A liquidez corrente do Economatica (AC/PC) inclui estoques e recebíveis, que não pagam dívida no curto prazo com a mesma certeza.

In [12]:
ea = base.adicionar_explicativas(ea)
risco = ea[ea['em_risco']]
expl = list(cfg.EXPLICATIVAS)
print('Na amostra em risco: sem dívida =', int(risco['sem_divida'].sum()))
risco[expl + ['idade_registro']].describe(percentiles=[.01, .25, .5, .75, .99]).T

Na amostra em risco: sem dívida = 57


,count,mean,std,min,1%,25%,50%,75%,99%,max
alavancagem,"1.704,000","0,308","0,889","0,000","0,000","0,154","0,287","0,403","0,749","36,163"
divida_cp_share,"1.647,000","0,310","0,226","0,000","0,017","0,145","0,244","0,418","1,000","1,000"
roa,"1.708,000","0,077","0,069","-0,146","0,000","0,037","0,064","0,100","0,303","1,104"
liquidez,"1.704,000","0,522","1,117","0,000","0,002","0,109","0,291","0,592","4,078","19,999"
fco_divida,"1.646,000","30,174","769,658","-51,561","-0,829","0,116","0,272","0,574","12,242","24.720,000"
log_ativo,"1.708,000","15,085","1,800","3,133","10,916","13,937","15,182","16,252","19,051","20,773"
idade_registro,"1.704,000","19,518","17,745","-10,000","-4,000","5,000","15,000","32,000","73,000","85,000"


**Leitura.** As caudas são longas. Há alavancagem acima de 1 (dívida maior que o ativo, ou seja, patrimônio líquido negativo) e `fco_divida` de milhares (empresas com dívida quase nula). Esses extremos dominariam uma regressão logística, por isso a winsorização da seção 8.

## 6. Baseline: Z''-score de Altman (mercados emergentes)

`Z'' = 3,25 + 6,56·X1 + 3,26·X2 + 6,72·X3 + 1,05·X4`, com:
- `X1 = (AC − PC)/AT` (capital de giro);
- `X2 = reservas de lucro/AT` (lucros retidos);
- `X3 = EBIT/AT` (rentabilidade operacional);
- `X4 = PL/(AT − PL)` (patrimônio sobre passivo).

Quanto **menor** o Z'', maior o risco. O X2 usa as reservas de lucro **com o sinal**: no Economatica, o valor negativo é o saldo de reservas menos prejuízos acumulados (decisão da Fase 1). O Z'' é só **comparação**. Ele usa EBIT (no X3), mas isso é aceitável aqui porque é um índice externo, de coeficientes fixos, e não é estimado nos nossos dados.

In [13]:
ea = base.adicionar_zscore(ea)
risco = ea[ea['em_risco']]
print('Reservas de lucro negativas na amostra em risco:', int((risco['reservas_lucro'] < 0).sum()))
risco[['z_x1', 'z_x2', 'z_x3', 'z_x4', 'z_score']].describe().T

Reservas de lucro negativas na amostra em risco: 13


,count,mean,std,min,25%,50%,75%,max
z_x1,"1.708,000","0,183","0,205","-2,076","0,048","0,160","0,309","0,924"
z_x2,"1.708,000","0,139","0,114","-0,243","0,055","0,120","0,199","0,803"
z_x3,"1.708,000","0,117","0,088","0,002","0,067","0,099","0,141","1,428"
z_x4,"1.708,000","1,099","1,454","-0,975","0,400","0,728","1,212","18,954"
z_score,"1.708,000","6,845","2,642","-10,860","5,311","6,342","7,706","28,583"


## 7. Divisão temporal

Os anos não são embaralhados. O modelo aprende com o passado e é avaliado no futuro, como seria usado na prática.

| split | t | os alvos olham |
|---|---|---|
| treino | 2010–2018 | até 2020 |
| validação (intervalo) | 2019–2020 | 2020–2022 |
| teste | 2021–2023 | 2022–2025 |

O **intervalo de dois anos** evita vazamento. O alvo de t=2018, o último do treino, usa 2019 e 2020. Se o teste começasse em 2019, o treino já teria "visto" o desempenho de 2019 e 2020 pelo alvo. A validação (2019–2020) serve para escolher hiperparâmetros e limiares na Fase 4. **O teste nunca é usado para escolher nada.**

In [14]:
ea = base.adicionar_split(ea)
risco = ea[ea['em_risco']]
s = risco.groupby('split', sort=False)['Y'].agg(empresa_anos='size', eventos='sum', taxa='mean').reindex(['treino', 'validacao', 'teste'])
s['eventos'] = s['eventos'].astype(int)
s

,empresa_anos,eventos,taxa
split,,,
treino,957,122,"0,127"
validacao,276,21,"0,076"
teste,475,54,"0,114"


## 8. Winsorização

Cada explicativa é cortada nos percentis 1% e 99%. Os **limites são calculados só no treino** (amostra em risco) e aplicados a todas as linhas. Se usássemos a base toda, a distribuição do teste vazaria para o treino. As colunas originais são mantidas, e as winsorizadas ganham o sufixo `_w`. Qualquer imputação ou padronização na Fase 4 seguirá a mesma regra.

O Z''-score **não** é winsorizado, porque é usado como fórmula fechada.

In [15]:
lim = base.limites_winsor(ea, expl)
ea = base.aplicar_winsor(ea, lim)
risco = ea[ea['em_risco']]
cortadas = pd.DataFrame({sp: [((g[c] < lo) | (g[c] > hi)).mean() for c, (lo, hi) in lim.iterrows()]
                         for sp, g in risco.groupby('split')}, index=lim.index)[['treino', 'validacao', 'teste']]
display(lim)
print('% de observações cortadas, por split:')
cortadas.style.format(lambda x: style.pct_br(x, 1))

,p_inf,p_sup
alavancagem,"0,000","0,730"
divida_cp_share,"0,026","0,996"
roa,"0,000","0,272"
liquidez,"0,003","6,774"
fco_divida,"-0,560","12,060"
log_ativo,"10,677","19,347"


% de observações cortadas, por split:


,treino,validacao,teste
alavancagem,"1,0%","1,1%","1,5%"
divida_cp_share,"2,1%","5,4%","3,8%"
roa,"2,1%","2,2%","2,5%"
liquidez,"2,1%","2,5%","1,3%"
fco_divida,"2,1%","2,9%","3,2%"
log_ativo,"2,1%","0,4%","0,8%"


**Leitura.** No treino, cerca de 2% das observações são cortadas por construção (1% em cada cauda). Se a validação ou o teste tivessem muito mais cortes, seria sinal de que a distribuição mudou ao longo do tempo. Isso é esperado em parte com a onda de IPOs, porque entram empresas novas, maiores e menos alavancadas.

## 9. Salvar

In [16]:
ea = ea.sort_values(['ticker', 'ano']).reset_index(drop=True)

# conferência: o pipeline passo a passo deste notebook é idêntico ao base.construir()
ref, _ = base.construir()
pd.testing.assert_frame_equal(ea[ref.columns], ref)

cfg.DATA_PROCESSED.mkdir(exist_ok=True)
ea.to_parquet(cfg.DATA_PROCESSED / 'base_empresa_ano.parquet', index=False)
base.escrever_dicionario(ea, cfg.DATA_PROCESSED / 'dicionario_variaveis.md')
print('Salvo:', ea.shape, '->', cfg.DATA_PROCESSED / 'base_empresa_ano.parquet')

Salvo: (3633, 58) -> C:\Users\mitsu\OneDrive\Documentos\GitHub\Projeto-2_CB\data\processed\base_empresa_ano.parquet


## Resumo da Fase 2

- **Base empresa-ano:** 3.633 empresa-anos de 284 empresas não financeiras (2010–2025).
- **Amostra em risco** (t = 2010–2023, fora de distress em t, alvo observado): 1.708 empresa-anos e **197 eventos (11,5%)**. Por split: treino 957 (122 eventos, 12,7%), validação 276 (21, 7,6%) e teste 475 (54, 11,4%).
- **O maior corte do funil é "já em distress em t"** (41% das linhas), porque a cobertura menor que 1 é comum com despesa financeira bruta e juros altos.
- **Apenas 12 linhas** se perdem por empresas que somem do painel. O viés de sobrevivência está nas empresas que **nunca entraram** na base (Fase 3).
- Arquivos: `data/processed/base_empresa_ano.parquet` e `data/processed/dicionario_variaveis.md`.